# 01. State 및 Schema 정의
LangGraph에서 사용할 데이터 모델과 Enum을 정의합니다.

In [ ]:
import operator
from enum import Enum
from typing import Annotated, Any, Dict, List, Optional
from pydantic import BaseModel, Field

class Domain(str, Enum):
    TECHNOLOGY = "technology"
    MARKET = "market"
    FINANCE = "finance"
    RISK = "risk"
    TEAM = "team"

class GateStatus(str, Enum):
    PASS = "PASS"
    HOLD_INSUFFICIENT_EVIDENCE = "HOLD_INSUFFICIENT_EVIDENCE"
    HOLD_CRITICAL_RISK = "HOLD_CRITICAL_RISK"
    HOLD_NO_DATA_RIGHTS = "HOLD_NO_DATA_RIGHTS"
    HOLD_MATERIAL_CONTRADICTION = "HOLD_MATERIAL_CONTRADICTION"
    HOLD_FINANCIAL_SURVIVAL = "HOLD_FINANCIAL_SURVIVAL"

class FinalDecision(str, Enum):
    RECOMMEND = "투자검토 추천"
    CONDITIONAL = "조건부 검토"
    HOLD_RAG = "근거 부족 판단 유보"
    HOLD_GATE = "투자 보류 및 추가 실사"
    HOLD_SCORE = "보류 (기준 미달)"


In [ ]:
class CompanySnapshot(BaseModel):
    name: str = Field(description="기업명")
    description: str = Field(default="", description="기업 기본 설명")
    is_unlisted: bool = Field(default=True)
    investment_stage: str = Field(default="Seed")
    has_exited: bool = Field(default=False)

class Question(BaseModel):
    domain: Domain
    item_id: str
    query: str

class Evidence(BaseModel):
    item_id: str
    content: str
    source_type: str
    url: Optional[str] = None
    evidence_grade: str = "U"
    is_direct: bool = False
    has_contradiction: bool = False

class GateResult(BaseModel):
    status: GateStatus = GateStatus.PASS
    reason: Optional[str] = None

class ProfileResult(BaseModel):
    profile_name: str
    profile_score: float
    pros: List[str] = Field(default_factory=list)
    cons: List[str] = Field(default_factory=list)
    questions_for_dd: List[str] = Field(default_factory=list)

class ConsensusResult(BaseModel):
    final_score: float
    score_std_dev: float
    is_high_disagreement: bool = False
    final_decision: FinalDecision
    summary_reason: str

class AgentState(BaseModel):
    current_company: Optional[CompanySnapshot] = None
    evaluation_questions: List[Question] = Field(default_factory=list)
    retrieved_evidence: List[Evidence] = Field(default_factory=list)
    validated_evidence: List[Evidence] = Field(default_factory=list)
    retry_count: Dict[str, int] = Field(default_factory=dict)
    unknown_items: List[str] = Field(default_factory=list)
    domain_scores: Dict[Domain, float] = Field(default_factory=dict)
    evidence_coverage: float = 0.0
    gate_result: Optional[GateResult] = None
    profile_results: Annotated[List[ProfileResult], operator.add] = Field(default_factory=list)
    consensus_result: Optional[ConsensusResult] = None
    report_errors: List[str] = Field(default_factory=list)
